In [1]:
'''
计算个人综合表现分
input:36个月内的所有比赛日期和成绩
output:个人综合表现分
'''
import datetime
from connect_sql import con_db,operateMysql,operateMysql_multiple
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
from loguru import logger
###输入日期字符串，返回距今过去了多少月
def month_diff(date_str):
    date_format = "%Y-%m-%d"
    input_date = datetime.datetime.strptime(date_str, date_format)
    current_date = datetime.datetime.now()
    delta = relativedelta(current_date, input_date)
    total_months = delta.years * 12 + delta.months
    return total_months
###根据月份差值返回权重
'''
距今月份
权重
0-11
1
12-17
0.995
18-23
0.99
24-29
0.985
30-35
0.98
'''
def get_weight(month_diff):
    if month_diff <= 11:
        return 1
    elif 12 <= month_diff <= 17:
        return 0.995
    elif 18 <= month_diff <= 23:
        return 0.99
    elif 24 <= month_diff <= 29:
        return 0.985
    elif 30 <= month_diff <= 35:
        return 0.98
    else:
        return 0
###对成绩进行加权处理，输出综合表现分
'''
第一场	第二场	第三场	第四场	第五场
0.97	0.99	1	1.01	1.02
        0.98	0.99	1	1.01
                0.99	1	1
                        0.99	1
                            0.99

'''
def calculate_weighted_score(weighted_scores):
    results = []
    n = len(weighted_scores)
    weight = [[0.97],[0.99,0.98],[1,0.99,0.99],[1.01,1,1,0.99],[1.02,1.01,1,1,0.99]]
    for i in range(n):
        result = []
        for j in range(i+1):
            result.append(weighted_scores[j]*weight[i][j])
        results.append(result)
    score = []
    for r in results:
        score.append(np.floor(np.mean(r)))
    

    return score


def deal_time_score(input_data):
    '''对所有成绩进行排序输出前5名'''
    input_data_sort = pd.DataFrame(input_data).sort_values(by='score',ascending=False).head(5)
    input_data_sort['month_diff'] = input_data_sort['date'].apply(month_diff)
    input_data_sort['weight'] = input_data_sort['month_diff'].apply(get_weight)
    input_data_sort['weighted_score'] = input_data_sort['score'] * input_data_sort['weight']
    weighted_scores = calculate_weighted_score(input_data_sort['weighted_score'].values)
    if len(weighted_scores) ==0:
        return 0
    else:
        return max (weighted_scores)
if __name__ == "__main__":
    # race_info = select_user_data(data_service_race_info_id)
    input_data = [
        {'date': '2026-01-24', 'score': 600},
        {'date': '2025-03-19', 'score': 550},
        {'date': '2024-12-18', 'score': 500},
        {'date': '2024-01-19', 'score': 450},
        {'date': '2023-05-31', 'score': 400}
    ]
    score = deal_time_score(input_data)
    print(f"个人综合表现分: {score}")


个人综合表现分: 582.0
